<a href="https://colab.research.google.com/github/alter-mix-dev/wasapcloudapi/blob/feat%2Fpedidos/AGENTE_PEDIDOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# BLOCK 1: PREPARACIÓN DEL ENTORNO Y CREDENCIALES
#En este bloque instalamos todas las librerías necesarias de Python y del sistema,
# importamos los módulos globales y cargamos de forma segura tus credenciales actualizadas desde Colab Secrets

In [ ]:
# 1.1 INSTALACIÓN DE DEPENDENCIAS
# ==============================================================================
# Instalamos componentes del sistema y librerías de Python requeridas en un solo paso
!apt-get update -qq
!apt-get install -y zstd -qq  # Requerido por Ollama
!curl -fsSL https://ollama.com | sh
!pip install ollama fastapi uvicorn pyngrok requests pytz --quiet

In [ ]:
# 1.2 IMPORTS GLOBALES Y CONFIGURACIÓN HORARIA
# ==============================================================================
import os
import time
import threading
import subprocess
import requests
from datetime import datetime
import pytz

import ollama
import uvicorn
from fastapi import FastAPI, Request, BackgroundTasks
from pyngrok import ngrok
from google.colab import userdata


In [ ]:
# 1.3 CARGA DE CREDENCIALES (COLAB SECRETS)
# ==============================================================================
VERIFY_TOKEN = "mi_token"  # Debe coincidir con el configurado en Meta for Developers
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')  # Tu nuevo token de 2 meses
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = "1061287516789190"  # ID de la cuenta de WhatsApp Business

print("✅ Entorno preparado e imports cargados.")
print("✅ Credenciales leídas desde Secrets correctamente.")

In [ ]:
#BLOCK 2: BACKEND DE IA (OLLAMA & MODELO)
#Este bloque se encarga de inicializar el servidor local de Ollama en la máquina virtual de Colab en segundo plano
# y descargar el modelo Llama 3.1.

In [ ]:
# 2.1 ARRANQUE DEL SERVIDOR OLLAMA
# ==============================================================================
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Tiempo de espera para levantar el servicio local

# Descarga del modelo específico
!ollama pull llama3.1:8b
print("✅ Servidor de Ollama y Llama 3.1 listos.")

In [ ]:
#BLOCK 3: LÓGICA DE NEGOCIO Y MODELO DE CONTEXTO
#Definimos la lógica del asistente del gimnasio. El modelo ahora recibe inyección dinámica del día y la hora de México
# para responder de manera coherente si el usuario pregunta por horarios o membresías.

In [ ]:
# 3.1 BASE DE DATOS EN MEMORIA
# ==============================================================================
mensajes_recibidos = []
usuarios_conocidos = set()  # Almacena los números que ya recibieron el saludo inicial

In [ ]:
# 3.2 HELPERS Y GENERACIÓN DE RESPUESTA CON CONTEXTO TEMPORAL
# ==============================================================================
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

def generar_respuesta(texto_usuario):
    # Inyección horaria de la CDMX para que el bot conozca el tiempo real
    zona_horaria = pytz.timezone('America/Mexico_City')
    ahora = datetime.now(zona_horaria)
    fecha_actual = ahora.strftime("%A, %d de %B de %Y")
    hora_actual = ahora.strftime("%H:%M")

    prompt = (
        f"Eres el asistente de atención a clientes de un gimnasio. "
        f"Hoy es {fecha_actual} y la hora actual es {hora_actual}. "
        f"Responde de forma breve (2-3 líneas) a la siguiente pregunta sobre horarios o membresías:\n\n"
        + texto_usuario
    )

    respuesta = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content": prompt}])
    return respuesta["message"]["content"]

print("✅ Helpers de lógica de negocio y generación de prompts configurados.")

In [ ]:
#BLOCK 4: DEFINICIÓN DEL SERVIDOR WEBHOOK (FASTAPI)
#Configuramos nuestra aplicación FastAPI con los endpoints requeridos por Meta. Procesamos los mensajes
#entrantes en segundo plano (BackgroundTasks) para mandar una confirmación HTTP inmediata a WhatsApp y evitar duplicidades.

In [ ]:
# 4.1 ENDPOINTS DE FASTAPI
# ==============================================================================
app = FastAPI()

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}  # Meta recibe el ACK inmediato


In [ ]:
# 4.2 ACCIONES DEL WEBHOOK
# ==============================================================================
def enviar_respuesta(numero, texto):
    url = f"https://facebook.com{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }
    requests.post(url, headers=headers, json=payload)

def procesar_y_responder(numero, texto):
    if numero not in usuarios_conocidos:
        usuarios_conocidos.add(numero)
        enviar_respuesta(numero, "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?")
    else:
        respuesta = generar_respuesta(texto)
        enviar_respuesta(numero, respuesta)

print("✅ Servidor Webhook declarado.")

In [ ]:
BLOCK 5: DESPLIEGUE, TÚNEL NGROK Y PRUEBAS
#Este bloque expone tu servidor local al internet público por medio de un túnel ngrok y lanza las simulaciones de prueba.
#Incluye el código muerto de tus diagnósticos y la prueba explícita para tu nuevo token.

In [ ]:
# 5.1 ACTIVACIÓN DE TÚNEL Y SERVIDOR
# ==============================================================================
ngrok.kill()  # Limpiamos cualquier túnel previo trabado
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)

print("🔗 URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook\n")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo para asegurar el arranque


In [ ]:
# 5.2 SIMULACIÓN DE FLUJO COMPLETO (LOCAL)
# ==============================================================================
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "Hola"}}
    ]}}]}]
}
r1 = requests.post("http://127.0.0", json=payload_1)
print("🧪 Simulación de primer mensaje (Saludo):", r1.status_code, r1.text)


In [ ]:
# 5.3 PRUEBA EXCLUSIVA PARA EL TOKEN DE 2 MESES (CONEXIÓN CON META)
# ==============================================================================
print("\n🔍 --- PROBANDO VALIDEZ DE TU NUEVO TOKEN DE 2 MESES ---")
# Intentamos vincular la app con el WABA ID usando el nuevo token de secrets
r_token = requests.post(
    f"https://facebook.com{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)

if r_token.status_code == 200:
    print("✅ ¡ÉXITO! Meta aceptó el token correctamente. Código 200 OK.")
    print("Detalle:", r_token.text)
else:
    print(f"❌ ERROR {r_token.status_code}: El token guardado en Secrets sigue dando problemas.")
    print("Respuesta de Meta:", r_token.text)

In [ ]:
# 5.4 CÓDIGO MUERTO DE PRUEBAS ADICIONALES Y DIAGNÓSTICOS ANTERIORES
# ==============================================================================
_ = '''
payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "¿A qué hora abren los sábados?"}}
    ]}}]}]
}
r2 = requests.post("http://127.0.0", json=payload_2)
print("Segundo mensaje:", r2.status_code, r2.text)
'''

_ = '''
if 'mensajes_recibidos' not in locals(): mensajes_recibidos = []
if 'respuestas_enviadas' not in locals(): respuestas_enviadas = 0
if 'errores_procesamiento' not in locals(): errores_procesamiento = 0
if 'conversaciones_por_numero' not in locals(): conversaciones_por_numero = {}

print("--- DIAGNÓSTICO DEL BOT ---")
print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", respuestas_enviadas)
print("Errores:", errores_procesamiento)
try: print("Llama Stack vivo:", proceso_llama_stack.poll() is None)
except NameError: print("Llama Stack vivo: Error (Variable no definida)")
try: print("Servidor vivo:", hilo.is_alive())
except NameError: print("Servidor vivo: Error (Variable no definida)")
'''

_ = '''
url = tunel_publico.public_url + "/webhook"
r = requests.get(url, params={"hub.verify_token": VERIFY_TOKEN, "hub.challenge": "12345"})
print("Prueba Túnel:", r.status_code, r.text)
'''